In [2]:
import gradio as gr
from datetime import datetime

# ==========================================
# 1. CAPA DE REGLAS / DOMINIO (POO)
# ==========================================

class EstadoCita:
    PROGRAMADA = "Programada"
    CANCELADA = "Cancelada"
    ATENDIDA = "Atendida"

class Servicio:
    def __init__(self, codigo: str, nombre: str, tarifa: float):
        self._codigo = codigo
        self._nombre = nombre
        self._tarifa = tarifa

    @property
    def codigo(self): return self._codigo
    @property
    def nombre(self): return self._nombre
    @property
    def tarifa(self): return self._tarifa

class Veterinario:
    def __init__(self, cedula: str, nombre: str, tarjeta_profesional: str):
        self._cedula = cedula
        self._nombre = nombre
        self._tarjeta_profesional = tarjeta_profesional

    @property
    def cedula(self): return self._cedula
    @property
    def nombre(self): return self._nombre
    @property
    def tarjeta_profesional(self): return self._tarjeta_profesional

class Mascota:
    def __init__(self, nombre: str, especie: str, raza: str, anio_nacimiento: int, peso: float):
        self._nombre = nombre
        self._especie = especie
        self._raza = raza
        self._anio_nacimiento = anio_nacimiento
        self._peso = peso

    @property
    def nombre(self): return self._nombre
    @property
    def especie(self): return self._especie
    @property
    def raza(self): return self._raza
    @property
    def anio_nacimiento(self): return self._anio_nacimiento
    @property
    def peso(self): return self._peso

class Propietario:
    def __init__(self, cedula: str, nombre: str, telefono: str, municipio: str):
        self._cedula = cedula
        self._nombre = nombre
        self._telefono = telefono
        self._municipio = municipio
        self._mascotas = []

    @property
    def cedula(self): return self._cedula
    @property
    def nombre(self): return self._nombre
    @property
    def telefono(self): return self._telefono
    @property
    def municipio(self): return self._municipio
    @property
    def mascotas(self): return self._mascotas

    def agregar_mascota(self, mascota: Mascota):
        self._mascotas.append(mascota)

class Factura:
    def __init__(self, codigo: str, servicios: list):
        self._codigo = codigo
        self._servicios = servicios
        self._valor_total = self.calcular_total()

    @property
    def codigo(self): return self._codigo
    @property
    def valor_total(self): return self._valor_total
    @property
    def servicios(self): return self._servicios

    def calcular_total(self) -> float:
        return sum(s.tarifa for s in self._servicios)

class Cita:
    def __init__(self, codigo: str, propietario: Propietario, mascota: Mascota, fecha: str, hora: str, veterinario: Veterinario, servicio_inicial: Servicio):
        self._codigo = codigo
        self._propietario = propietario
        self._mascota = mascota
        self._fecha = fecha
        self._hora = hora
        self._veterinario = veterinario
        self._servicios = [servicio_inicial]
        self._estado = EstadoCita.PROGRAMADA
        self._factura = None

    @property
    def codigo(self): return self._codigo
    @property
    def fecha(self): return self._fecha
    @property
    def hora(self): return self._hora
    @property
    def propietario(self): return self._propietario
    @property
    def mascota(self): return self._mascota
    @property
    def veterinario(self): return self._veterinario
    @property
    def servicios(self): return self._servicios
    @property
    def estado(self): return self._estado
    @estado.setter
    def estado(self, nuevo_estado): self._estado = nuevo_estado
    @property
    def factura(self): return self._factura
    @factura.setter
    def factura(self, nueva_factura): self._factura = nueva_factura

    def agregar_servicio(self, servicio: Servicio):
        self._servicios.append(servicio)

class Clinica:
    def __init__(self, nit: str, nombre: str, municipio: str, codigo_dane: str, departamento: str, direccion: str, barrio: str):
        self._nit = nit
        self._nombre = nombre
        self._municipio = municipio
        self._codigo_dane = codigo_dane
        self._departamento = departamento
        self._direccion = direccion
        self._barrio = barrio
        self._veterinarios = []
        self._servicios = []
        self._propietarios = []
        self._citas = []
        self._facturas = []

    @property
    def nit(self): return self._nit
    @property
    def nombre(self): return self._nombre
    @property
    def veterinarios(self): return self._veterinarios
    @property
    def servicios(self): return self._servicios
    @property
    def propietarios(self): return self._propietarios
    @property
    def citas(self): return self._citas

    def registrar_veterinario(self, vet: Veterinario) -> tuple[bool, str]:
        if any(v.cedula == vet.cedula for v in self._veterinarios):
            return False, f"Ya existe un veterinario con cédula {vet.cedula} en esta clínica."
        self._veterinarios.append(vet)
        return True, "Veterinario registrado con éxito."

    def registrar_servicio(self, serv: Servicio) -> tuple[bool, str]:
        if any(s.codigo == serv.codigo for s in self._servicios):
            return False, f"Ya existe un servicio con el código {serv.codigo} en esta clínica."
        self._servicios.append(serv)
        return True, "Servicio registrado con éxito."

    def registrar_propietario(self, prop: Propietario) -> tuple[bool, str]:
        if any(p.cedula == prop.cedula for p in self._propietarios):
            return False, f"Ya existe un propietario con la cédula {prop.cedula} en esta clínica."
        self._propietarios.append(prop)
        return True, "Propietario registrado con éxito."

    def programar_cita(self, cita: Cita) -> tuple[bool, str]:
        for c in self._citas:
            if c.estado == EstadoCita.PROGRAMADA and c.fecha == cita.fecha and c.hora == cita.hora:
                if c.veterinario.cedula == cita.veterinario.cedula:
                    return False, f"La veterinaria/o {c.veterinario.nombre} ya tiene una cita asignada el {cita.fecha} a las {cita.hora}."
                if c.propietario.cedula == cita.propietario.cedula and c.mascota.nombre.lower() == cita.mascota.nombre.lower():
                    return False, f"La mascota {cita.mascota.nombre} ya tiene una cita agendada el {cita.fecha} a las {cita.hora}."

        self._citas.append(cita)
        return True, f"Cita {cita.codigo} programada exitosamente."

    def cancelar_cita(self, codigo_cita: str) -> tuple[bool, str]:
        cita = next((c for c in self._citas if c.codigo == codigo_cita), None)
        if not cita:
            return False, f"Error: No se encontró la cita {codigo_cita}."
        if cita.estado != EstadoCita.PROGRAMADA:
            return False, f"Operación rechazada: La cita {codigo_cita} ya fue {cita.estado.lower()} y no se puede cancelar."

        cita.estado = EstadoCita.CANCELADA
        return True, f"Cita {codigo_cita} cancelada exitosamente."

    def atender_cita(self, codigo_cita: str, codigo_factura: str, codigos_servicios_extra: list) -> tuple[bool, str]:
        cita = next((c for c in self._citas if c.codigo == codigo_cita), None)
        if not cita:
            return False, f"Error: No se encontró la cita {codigo_cita}."
        if cita.estado != EstadoCita.PROGRAMADA:
            return False, f"Operación rechazada: No se puede atender la cita {codigo_cita} porque se encuentra en estado '{cita.estado}'."

        for cod in codigos_servicios_extra:
            serv = next((s for s in self._servicios if s.codigo == cod), None)
            if serv:
                cita.agregar_servicio(serv)

        cita.estado = EstadoCita.ATENDIDA
        factura = Factura(codigo_factura, cita.servicios)
        cita.factura = factura
        self._facturas.append(factura)
        return True, f"Cita {codigo_cita} Atendida. Factura {codigo_factura} expedida por un total de ${factura.valor_total:,.0f}."


# ==========================================
# 2. CAPA DE PRESENTACIÓN / CONTROLADOR
# ==========================================

class Presentacion:
    def __init__(self):
        self.clinicas = []

    def buscar_clinica(self, nit: str) -> Clinica:
        return next((c for c in self.clinicas if c.nit == nit), None)

    def precargar_datos(self) -> str:
        if self.buscar_clinica("901456789-1"):
            return "Los datos de prueba ya fueron cargados previamente."

        # Clínica Huellitas
        huel = Clinica("901456789-1", "Clínica Veterinaria Huellitas", "San Gil", "68679", "Santander", "Calle 12 # 9-40", "Centro")
        huel.registrar_veterinario(Veterinario("63512478", "Juliana Ortiz Ramírez", "TP-18452"))
        huel.registrar_veterinario(Veterinario("91254863", "Mauricio Pinzón Gómez", "TP-20117"))
        huel.registrar_servicio(Servicio("CG", "Consulta general", 60000.0))
        huel.registrar_servicio(Servicio("VAC", "Vacunación", 45000.0))
        huel.registrar_servicio(Servicio("DES", "Desparasitación", 30000.0))
        huel.registrar_servicio(Servicio("PEL", "Baño y peluquería", 50000.0))
        self.clinicas.append(huel)

        # Clínica Patitas Felices
        pati = Clinica("900987654-3", "Clínica Veterinaria Patitas Felices", "Floridablanca", "68276", "Santander", "Carrera 27 # 30-15", "Cañaveral")
        pati.registrar_veterinario(Veterinario("13745821", "Sergio Arenas Villamizar", "TP-15893"))
        pati.registrar_veterinario(Veterinario("37896542", "Diana Carolina Rey Moreno", "TP-22406"))
        pati.registrar_veterinario(Veterinario("1098654321", "Felipe Castellanos Díaz", "TP-31578"))
        pati.registrar_servicio(Servicio("CG", "Consulta general", 65000.0))
        pati.registrar_servicio(Servicio("VAC", "Vacunación", 48000.0))
        pati.registrar_servicio(Servicio("DES", "Desparasitación", 32000.0))
        pati.registrar_servicio(Servicio("ECO", "Ecografía", 120000.0))
        self.clinicas.append(pati)

        return "Datos de prueba (Huellitas y Patitas Felices) cargados correctamente."

    def registrar_clinica(self, nit, nombre, municipio, dane, dpto, direccion, barrio):
        if self.buscar_clinica(nit):
            return f"[RECHAZADO] Ya existe una clínica registrada con el NIT {nit}."
        nueva = Clinica(nit, nombre, municipio, dane, dpto, direccion, barrio)
        self.clinicas.append(nueva)
        return f"[ÉXITO] Clínica '{nombre}' registrada exitosamente."

    def registrar_veterinario(self, nit, cedula, nombre, tarjeta):
        cli = self.buscar_clinica(nit)
        if not cli: return "[ERROR] Clínica no encontrada."
        exito, msj = cli.registrar_veterinario(Veterinario(cedula, nombre, tarjeta))
        return msj

    def registrar_servicio(self, nit, codigo, nombre, tarifa):
        cli = self.buscar_clinica(nit)
        if not cli: return "[ERROR] Clínica no encontrada."
        exito, msj = cli.registrar_servicio(Servicio(codigo, nombre, float(tarifa or 0)))
        return msj

    def programar_cita(self, nit, cod_cita, ced_prop, nom_prop, tel_prop, mun_prop, nom_mascota, esp_mascota, raza_mascota, anio_mascota, peso_mascota, fecha, hora, ced_vet, cod_serv):
        cli = self.buscar_clinica(nit)
        if not cli: return "[ERROR] Clínica no encontrada."

        vet = next((v for v in cli.veterinarios if v.cedula == ced_vet), None)
        if not vet: return f"[ERROR] Veterinario con cédula {ced_vet} no existe en esta clínica."

        serv = next((s for s in cli.servicios if s.codigo == cod_serv), None)
        if not serv: return f"[ERROR] Servicio con código {cod_serv} no existe en esta clínica."

        prop = next((p for p in cli.propietarios if p.cedula == ced_prop), None)
        if not prop:
            prop = Propietario(ced_prop, nom_prop, tel_prop, mun_prop)
            cli.registrar_propietario(prop)

        masc = next((m for m in prop.mascotas if m.nombre.lower() == nom_mascota.lower()), None)
        if not masc:
            masc = Mascota(nom_mascota, esp_mascota, raza_mascota, int(anio_mascota or 2023), float(peso_mascota or 0))
            prop.agregar_mascota(masc)

        cita = Cita(cod_cita, prop, masc, fecha, hora, vet, serv)
        exito, msj = cli.programar_cita(cita)
        return msj

    def cancelar_cita(self, nit, cod_cita):
        cli = self.buscar_clinica(nit)
        if not cli: return "[ERROR] Clínica no encontrada."
        exito, msj = cli.cancelar_cita(cod_cita)
        return msj

    def atender_cita(self, nit, cod_cita, cod_fac, servs_extra_str):
        cli = self.buscar_clinica(nit)
        if not cli: return "[ERROR] Clínica no encontrada."
        extras = [s.strip() for s in servs_extra_str.split(",") if s.strip()]
        exito, msj = cli.atender_cita(cod_cita, cod_fac, extras)
        return msj

    def consultar_citas_fecha(self, nit, fecha):
        cli = self.buscar_clinica(nit)
        if not cli: return "[ERROR] Clínica no encontrada."
        citas_f = [c for c in cli.citas if c.fecha == fecha]
        if not citas_f: return f"No hay citas registradas en la clínica para la fecha {fecha}."
        res = [f"--- Citas en {cli.nombre} ({fecha}) ---"]
        for c in citas_f:
            res.append(f"Código: {c.codigo} | Mascota: {c.mascota.nombre} | Estado: {c.estado}")
        return "\n".join(res)

    def consultar_historial_mascota(self, nit, nom_mascota):
        cli = self.buscar_clinica(nit)
        if not cli: return "[ERROR] Clínica no encontrada."
        citas_m = [c for c in cli.citas if c.mascota.nombre.lower() == nom_mascota.lower()]
        if not citas_m: return f"No se encontró historial para la mascota '{nom_mascota}'."
        res = [f"--- Historial de citas para la mascota '{nom_mascota}' ---"]
        for c in citas_m:
            res.append(f"Código: {c.codigo} | Fecha: {c.fecha} {c.hora} | Estado: {c.estado}")
        return "\n".join(res)


# ==========================================
# 3. INTERFAZ GRÁFICA CON GRADIO
# ==========================================

presentacion = Presentacion()

with gr.Blocks(title="Gestión de Citas Veterinarias") as app:
    gr.Markdown("# 🐾 Sistema de Gestión de Citas Veterinarias")

    btn_precargar = gr.Button("🚀 Cargar Datos de Prueba (Huellitas y Patitas Felices)", variant="primary")
    out_precargar = gr.Textbox(label="Estado de Carga Inicial")
    btn_precargar.click(presentacion.precargar_datos, outputs=out_precargar)

    with gr.Tabs():
        with gr.Tab("1. Registrar Clínica"):
            nit_reg = gr.Textbox(label="NIT (ej. 901456789-1)")
            nom_reg = gr.Textbox(label="Nombre de la Clínica")
            mun_reg = gr.Textbox(label="Municipio")
            dane_reg = gr.Textbox(label="Código DANE")
            dpto_reg = gr.Textbox(label="Departamento")
            dir_reg = gr.Textbox(label="Dirección")
            barrio_reg = gr.Textbox(label="Barrio")
            btn_reg = gr.Button("Registrar Clínica")
            out_reg = gr.Textbox(label="Resultado")
            btn_reg.click(presentacion.registrar_clinica,
                          inputs=[nit_reg, nom_reg, mun_reg, dane_reg, dpto_reg, dir_reg, barrio_reg],
                          outputs=out_reg)

        with gr.Tab("2. Registrar Vet / Servicio"):
            gr.Markdown("### Registrar Veterinario")
            nit_v = gr.Textbox(label="NIT Clínica")
            ced_v = gr.Textbox(label="Cédula Veterinario")
            nom_v = gr.Textbox(label="Nombre Completo")
            tp_v = gr.Textbox(label="Tarjeta Profesional")
            btn_v = gr.Button("Registrar Veterinario")
            out_v = gr.Textbox(label="Resultado")
            btn_v.click(presentacion.registrar_veterinario, inputs=[nit_v, ced_v, nom_v, tp_v], outputs=out_v)

            gr.Markdown("---")
            gr.Markdown("### Registrar Servicio")
            nit_s = gr.Textbox(label="NIT Clínica")
            cod_s = gr.Textbox(label="Código Servicio (ej. CG, VAC)")
            nom_s = gr.Textbox(label="Nombre Servicio")
            tar_s = gr.Number(label="Tarifa ($)")
            btn_s = gr.Button("Registrar Servicio")
            out_s = gr.Textbox(label="Resultado")
            btn_s.click(presentacion.registrar_servicio, inputs=[nit_s, cod_s, nom_s, tar_s], outputs=out_s)

        with gr.Tab("3. Programar Cita"):
            nit_p = gr.Textbox(label="NIT Clínica", value="901456789-1")
            cod_cit = gr.Textbox(label="Código Cita", value="CIT-001")
            with gr.Row():
                ced_p = gr.Textbox(label="Cédula Propietario", value="1098765432")
                nom_p = gr.Textbox(label="Nombre Propietario", value="Laura Méndez Plata")
                tel_p = gr.Textbox(label="Teléfono", value="3157894561")
                mun_p = gr.Textbox(label="Municipio Propietario", value="Bucaramanga")
            with gr.Row():
                nom_m = gr.Textbox(label="Nombre Mascota", value="Rocky")
                esp_m = gr.Textbox(label="Especie", value="Perro")
                raz_m = gr.Textbox(label="Raza", value="Criolla")
                ani_m = gr.Number(label="Año Nacimiento", value=2023)
                pes_m = gr.Number(label="Peso (kg)", value=18)
            with gr.Row():
                fec_p = gr.Textbox(label="Fecha (DD/MM/AAAA)", value="10/11/2026")
                hor_p = gr.Textbox(label="Hora (HH:MM)", value="09:00")
                vet_p = gr.Textbox(label="Cédula Veterinario", value="63512478")
                ser_p = gr.Textbox(label="Código Servicio Inicial", value="CG")
            btn_p = gr.Button("Programar Cita")
            out_p = gr.Textbox(label="Resultado")
            btn_p.click(presentacion.programar_cita,
                        inputs=[nit_p, cod_cit, ced_p, nom_p, tel_p, mun_p, nom_m, esp_m, raz_m, ani_m, pes_m, fec_p, hor_p, vet_p, ser_p],
                        outputs=out_p)

        with gr.Tab("4. Cancelar Cita"):
            nit_c = gr.Textbox(label="NIT Clínica", value="901456789-1")
            cod_cit_c = gr.Textbox(label="Código Cita a Cancelar", value="CIT-001")
            btn_c = gr.Button("Cancelar Cita", variant="stop")
            out_c = gr.Textbox(label="Resultado")
            btn_c.click(presentacion.cancelar_cita, inputs=[nit_c, cod_cit_c], outputs=out_c)

        with gr.Tab("5. Atender Cita / Facturar"):
            nit_a = gr.Textbox(label="NIT Clínica", value="901456789-1")
            cod_cit_a = gr.Textbox(label="Código Cita", value="CIT-001")
            cod_fac = gr.Textbox(label="Código Factura", value="FAC-001")
            ext_a = gr.Textbox(label="Servicios adicionales (códigos separados por coma)", value="VAC")
            btn_a = gr.Button("Atender y Facturar")
            out_a = gr.Textbox(label="Resultado")
            btn_a.click(presentacion.atender_cita, inputs=[nit_a, cod_cit_a, cod_fac, ext_a], outputs=out_a)

        with gr.Tab("6. Consultas"):
            gr.Markdown("### Consultar Citas por Fecha")
            nit_q1 = gr.Textbox(label="NIT Clínica", value="901456789-1")
            fec_q1 = gr.Textbox(label="Fecha (DD/MM/AAAA)", value="10/11/2026")
            btn_q1 = gr.Button("Consultar Citas")
            out_q1 = gr.Textbox(label="Resultado")
            btn_q1.click(presentacion.consultar_citas_fecha, inputs=[nit_q1, fec_q1], outputs=out_q1)

            gr.Markdown("---")
            gr.Markdown("### Historial de Mascota")
            nit_q2 = gr.Textbox(label="NIT Clínica", value="900987654-3")
            nom_q2 = gr.Textbox(label="Nombre Mascota", value="Luna")
            btn_q2 = gr.Button("Consultar Historial")
            out_q2 = gr.Textbox(label="Resultado")
            btn_q2.click(presentacion.consultar_historial_mascota, inputs=[nit_q2, nom_q2], outputs=out_q2)

if __name__ == "__main__":
    app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://acb1dce6bf8dbb62fa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
